Running importance analysis with Python API (Regression)
=========================================================

This notebook demonstrates running *VariantSpark* importance analysis using a **regression** random forest (`RandomForestRegressor`), as opposed to the classification variant.

`chr22-values.csv` contains a continuous response variable constructed as a weighted linear combination of two genomic variants:

$$\text{response} = 0.4 \times \texttt{22\_16051347\_G\_C} + (-0.6) \times \texttt{22\_16050984\_A\_G}$$

We would therefore expect these two positions to rank as the most important variables in the analysis.


Step 1: Create a spark session with VariantSpark jar attached.

In [1]:
import varspark as vs
from pyspark.sql import SparkSession 
spark = vs.configure_spark(
    SparkSession.builder.config('spark.jars', vs.find_jar())
).getOrCreate()

26/04/21 15:39:06 WARN Utils: Your hostname, RADON-BH resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/21 15:39:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/04/21 15:39:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 15:39:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Step 2: Create a `VarsparkContext` using `SparkSession` object (here injected as `spark`):

In [2]:
vc = vs.VarsparkContext(spark, silent = True)

Step 3: Load the features `fs` and continuous response `rs`.  and the `chr22-values.csv` file.

In [3]:
fs = vc.import_vcf('../../data/chr22_1000.vcf')
rs = vc.load_response('../../data/chr22-values.csv', 'response')

../../data/chr22_1000.vcf is loading to spark RDD, isBGZFile: false


Step 4: Fit a regression RF model using `RandomForestRegressor`.

In [4]:
rf = vs.RandomForestRegressor(vc, mtry_fraction=0.2, min_node_size=5, max_depth=10, seed=13)
rf.fit_trees(fs, rs, n_trees=500, batch_size=20)

Step 5: Retrieve top important variables and the full importance table as a pandas DataFrame:

In [5]:
ia = rf.importance_analysis()
top_variables = ia.important_variables(limit=10, normalized=True)

In [6]:
# return pandas dataframe of variable importances
importance = ia.variable_importance(normalized = True)
# sort by importance
importance = importance.sort_values('importance', ascending=False)
importance.head(10)

,variant_id,importance,splitCount
253,22_16051347_G_C,0.348393,1006
969,22_16051497_A_G,0.333925,957
975,22_16053791_C_A,0.096644,365
1229,22_16052239_A_G,0.045280,240
1343,22_16052513_G_C,0.028310,133
1113,22_16051453_A_C,0.016638,173
1,22_16052618_G_A,0.016545,99
384,22_16051249_T_C,0.016484,163
1669,22_16053862_C_T,0.014904,163
1797,22_16053659_A_C,0.014881,147


Step 6: Print the top variables by importance.

In [7]:
print("%s\t%s" % ('Variable', 'Importance'))
for var_and_imp in top_variables.values:
    print("%s\t%s" % tuple(var_and_imp))

Variable	Importance
22_16051347_G_C	0.3483927704755344
22_16051497_A_G	0.3339253542100561
22_16053791_C_A	0.09664432973185702
22_16052239_A_G	0.04527971665743685
22_16052513_G_C	0.028310118898845667
22_16051453_A_C	0.01663833664626107
22_16052618_G_A	0.016545262186734706
22_16051249_T_C	0.016484478312273444
22_16053862_C_T	0.014904156198159805
22_16053659_A_C	0.014880509047460955


For more information on using *VariantSpark* and the Python API please visit the [documentation](http://variantspark.readthedocs.io/en/latest/).